In [2]:
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv("adult.data.csv")

In [6]:
data.head()

,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
0,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
1,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
2,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
3,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
4,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


In [7]:
# DATA SET
cols = ["age","workclass","fnlwgt","education","education-num",
        "marital-status","occupation","relationship","race","sex",
        "capital-gain","capital-loss","hours-per-week","native-country","income"]

data = pd.read_csv("adult.data.csv", names=cols, na_values=" ?", skipinitialspace=True)

In [8]:
data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [9]:
# Remove missing values

data = data.dropna()

data["income"] = data["income"].apply(lambda x: 1 if x == ">50K" else 0)

data = pd.get_dummies(data)      # Convert text column into numbers
data = data.astype(float)

X = data.drop("income", axis=1).values
Y = data["income"].values.reshape(1, -1)

X = (X - X.mean()) / X.std()
X = X.T

print("data ready:", X.shape)

data ready: (108, 32561)


In [10]:
# Activation functions

def relu(x):
    return np.maximum(0, x)

def drelu(x):
    return x > 0

def sigmoid(x):
    return 1/(1+np.exp(-x))

In [11]:
# Initialize weights

np.random.seed(1)

w1 = np.random.randn(64, X.shape[0]) * np.sqrt(2/X.shape[0])
b1 = np.zeros((64,1))

w2 = np.random.randn(32, 64) * np.sqrt(2/64)
b2 = np.zeros((32,1))

w3 = np.random.randn(1, 32) * np.sqrt(2/32)
b3 = np.zeros((1,1))

In [12]:
# Training

lr = 0.01

for i in range(1001):

 # forward propagations   

    z1 = np.dot(w1, X) + b1
    a1 = relu(z1)
    z2 = np.dot(w2, a1) + b2
    a2 = relu(z2)
    z3 = np.dot(w3, a2) + b3
    a3 = sigmoid(z3)
    
    m = Y.shape[1]
    loss = -np.sum(Y*np.log(a3 + 1e-8) + (1-Y)*np.log(1-a3 + 1e-8)) / m
    
# backpropagations
    dz3 = a3 - Y
    dw3 = np.dot(dz3, a2.T) / m
    db3 = np.sum(dz3, axis=1, keepdims=True) / m
    da2 = np.dot(w3.T, dz3)
    dz2 = da2 * drelu(z2)
    dw2 = np.dot(dz2, a1.T) / m
    db2 = np.sum(dz2, axis=1, keepdims=True) / m
    da1 = np.dot(w2.T, dz2)
    dz1 = da1 * drelu(z1)
    dw1 = np.dot(dz1, X.T) / m
    db1 = np.sum(dz1, axis=1, keepdims=True) / m
    
 # update weights       
    w1 -= lr * dw1
    b1 -= lr * db1
    w2 -= lr * dw2
    b2 -= lr * db2
    w3 -= lr * dw3
    b3 -= lr * db3
    if i % 100 == 0:
        print("step", i, "loss:", loss)
        

step 0 loss: 1.1784099121817748
step 100 loss: 0.562465170437401
step 200 loss: 0.556693200918671
step 300 loss: 0.5516697240304304
step 400 loss: 0.5455901948550392
step 500 loss: 0.542806732148265
step 600 loss: 0.5406634661520999
step 700 loss: 0.5389503468814827
step 800 loss: 0.5375273210660124
step 900 loss: 0.5363040831780532
step 1000 loss: 0.533924095269425


In [13]:
# Accuracy

out = sigmoid(np.dot(w3, relu(np.dot(w2, relu(np.dot(w1, X) + b1)) + b2)) + b3)
pred = (out > 0.5).astype(int)

acc = np.mean(pred == Y) * 100
print("accuracy:", acc, "%")

accuracy: 76.38586038512331 %
